# Notebook 06 — VaR Backtesting and Out-of-Sample Validation

## Objective

This notebook evaluates the frozen candidate portfolios strictly out of
sample from 3 January 2022 onward.

Portfolio weights are not re-estimated using OOS information.

Historical and Parametric Normal VaR forecasts are generated using a rolling
500-day estimation window.

Model validity is assessed using:

- exceedance frequencies;
- Kupiec's proportion-of-failures test;
- Christoffersen's independence test;
- conditional coverage; and
- the Basel 250-day traffic-light framework.

The purpose is to determine whether the risk models developed earlier remain
reliable when confronted with previously unseen market observations.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import chi2, norm

In [2]:
PROJECT_ROOT = Path("..")

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

EXTERNAL_DIR = (
    PROJECT_ROOT
    / "data"
    / "external"
)

TABLE_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
)

FIGURE_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "figures"
)

RETURN_FILE = (
    PROCESSED_DIR
    / "simple_returns.csv"
)

SHORTLIST_FILE = (
    TABLE_DIR
    / "risk_shortlist_development.csv"
)

RF_FILE = (
    EXTERNAL_DIR
    / "DGS3MO.csv"
)

print(
    "Returns file exists:",
    RETURN_FILE.exists()
)

print(
    "Shortlist file exists:",
    SHORTLIST_FILE.exists()
)

print(
    "Risk-free file exists:",
    RF_FILE.exists()
)

Returns file exists: True
Shortlist file exists: True
Risk-free file exists: True


In [3]:
simple_returns = pd.read_csv(
    RETURN_FILE,
    index_col=0,
    parse_dates=True
).sort_index()

TICKERS = [
    "SPY",
    "EFA",
    "EEM",
    "AGG",
    "GLD",
    "XLE",
    "XLK"
]

simple_returns = (
    simple_returns[
        TICKERS
    ]
    .copy()
)

print(
    "Full return sample start:",
    simple_returns.index.min()
)

print(
    "Full return sample end:",
    simple_returns.index.max()
)

print(
    "Total observations:",
    len(simple_returns)
)

Full return sample start: 2004-11-19 00:00:00
Full return sample end: 2026-10-19 00:00:00
Total observations: 5512


In [4]:
TRAIN_END = pd.Timestamp(
    "2021-12-31"
)

OOS_START = pd.Timestamp(
    "2022-01-01"
)

returns_train = (
    simple_returns
    .loc[:TRAIN_END]
    .copy()
)

returns_oos = (
    simple_returns
    .loc[OOS_START:]
    .copy()
)

print(
    "Development observations:",
    len(returns_train)
)

print(
    "Development end:",
    returns_train.index.max()
)

print()

print(
    "OOS observations:",
    len(returns_oos)
)

print(
    "OOS start:",
    returns_oos.index.min()
)

print(
    "OOS end:",
    returns_oos.index.max()
)

Development observations: 4309
Development end: 2021-12-31 00:00:00

OOS observations: 1203
OOS start: 2022-01-03 00:00:00
OOS end: 2026-10-19 00:00:00


In [5]:
assert (
    returns_train.index.max()
    <
    returns_oos.index.min()
)

assert (
    returns_train.index.max()
    <= TRAIN_END
)

assert (
    returns_oos.index.min()
    >= OOS_START
)

print(
    "Development and OOS samples are correctly separated."
)

Development and OOS samples are correctly separated.


In [6]:
shortlist = pd.read_csv(
    SHORTLIST_FILE
)

print(
    "Number of shortlisted portfolios:",
    len(shortlist)
)

shortlist[
    [
        "Portfolio Label",
        "N",
        "Subset",
        "Method"
    ]
]

Number of shortlisted portfolios: 5


,Portfolio Label,N,Subset,Method
0,Equal Weight Benchmark,7,"SPY, EFA, EEM, AGG, GLD, XLE, XLK",Equal Weight
1,Best GMVP,2,"SPY, AGG",GMVP
2,Best Maximum Sharpe,3,"AGG, GLD, XLK",Maximum Sharpe
3,Best Risk Parity,3,"AGG, GLD, XLK",Risk Parity
4,Best Minimum ES,2,"SPY, AGG",Minimum ES


In [7]:
def parse_weights(
    value
):

    if isinstance(
        value,
        np.ndarray
    ):
        return value

    return np.array(
        [
            float(x)
            for x in str(value).split(";")
        ]
    )

In [8]:
if (
    "Weights Array"
    in shortlist.columns
):

    shortlist[
        "Frozen Weights"
    ] = (
        shortlist[
            "Weights Array"
        ]
        .apply(
            parse_weights
        )
    )

else:

    shortlist[
        "Frozen Weights"
    ] = (
        shortlist[
            "Weights"
        ]
        .apply(
            parse_weights
        )
    )

In [9]:
portfolio_shortlist_export = (
    shortlist[
        [
            "Portfolio Label",
            "N",
            "Subset",
            "Method"
        ]
    ]
    .copy()
)

portfolio_shortlist_export[
    "Weights"
] = (
    shortlist[
        "Frozen Weights"
    ]
    .apply(
        lambda w:
        str(
            [
                float(x)
                for x in w
            ]
        )
    )
)

portfolio_shortlist_export.to_csv(
    TABLE_DIR
    / "portfolio_shortlist.csv",
    index=False
)

print(
    "Exact frozen shortlist saved to:",
    TABLE_DIR
    / "portfolio_shortlist.csv"
)

portfolio_shortlist_export

Exact frozen shortlist saved to: ..\outputs\tables\portfolio_shortlist.csv


,Portfolio Label,N,Subset,Method,Weights
0,Equal Weight Benchmark,7,"SPY, EFA, EEM, AGG, GLD, XLE, XLK",Equal Weight,"[0.1428571429, 0.1428571429, 0.1428571429, 0.1..."
1,Best GMVP,2,"SPY, AGG",GMVP,"[0.0787430197, 0.9212569803]"
2,Best Maximum Sharpe,3,"AGG, GLD, XLK",Maximum Sharpe,"[0.6522812131, 0.1292391967, 0.2184795901]"
3,Best Risk Parity,3,"AGG, GLD, XLK",Risk Parity,"[0.6545410719, 0.1779072155, 0.1675517126]"
4,Best Minimum ES,2,"SPY, AGG",Minimum ES,"[0.0733156417, 0.9266843583]"


In [10]:
for _, row in shortlist.iterrows():

    assets = [
        x.strip()
        for x in row[
            "Subset"
        ].split(",")
    ]

    weights = row[
        "Frozen Weights"
    ]

    print(
        row[
            "Portfolio Label"
        ]
    )

    print(
        "Assets:",
        assets
    )

    print(
        "Weights:",
        np.round(
            weights,
            4
        )
    )

    print(
        "Weight sum:",
        round(
            weights.sum(),
            8
        )
    )

    print()

Equal Weight Benchmark


Assets: ['SPY', 'EFA', 'EEM', 'AGG', 'GLD', 'XLE', 'XLK']
Weights: [0.1429 0.1429 0.1429 0.1429 0.1429 0.1429 0.1429]
Weight sum: 1.0

Best GMVP
Assets: ['SPY', 'AGG']
Weights: [0.0787 0.9213]
Weight sum: 1.0

Best Maximum Sharpe
Assets: ['AGG', 'GLD', 'XLK']
Weights: [0.6523 0.1292 0.2185]
Weight sum: 1.0

Best Risk Parity
Assets: ['AGG', 'GLD', 'XLK']
Weights: [0.6545 0.1779 0.1676]
Weight sum: 1.0

Best Minimum ES
Assets: ['SPY', 'AGG']
Weights: [0.0733 0.9267]
Weight sum: 1.0



In [11]:
for _, row in shortlist.iterrows():

    assets = [
        x.strip()
        for x in row[
            "Subset"
        ].split(",")
    ]

    weights = row[
        "Frozen Weights"
    ]

    assert (
        len(assets)
        ==
        len(weights)
    )

    assert np.isclose(
        weights.sum(),
        1.0,
        atol=1e-6
    )

    assert np.all(
        weights >= 0
    )

print(
    "All frozen portfolio weights are valid."
)

All frozen portfolio weights are valid.


In [12]:
def portfolio_returns_from_weights(
    assets,
    weights,
    return_data
):

    subset_returns = (
        return_data[
            assets
        ]
    )

    portfolio_returns = (
        subset_returns.values
        @ weights
    )

    return pd.Series(
        portfolio_returns,
        index=subset_returns.index
    )

In [13]:
oos_portfolio_returns = (
    pd.DataFrame(
        index=returns_oos.index
    )
)

for _, row in shortlist.iterrows():

    assets = [
        x.strip()
        for x in row[
            "Subset"
        ].split(",")
    ]

    weights = row[
        "Frozen Weights"
    ]

    label = row[
        "Portfolio Label"
    ]

    oos_portfolio_returns[
        label
    ] = (
        portfolio_returns_from_weights(
            assets,
            weights,
            returns_oos
        )
    )


oos_portfolio_returns.head()

,Equal Weight Benchmark,Best GMVP,Best Maximum Sharpe,Best Risk Parity,Best Minimum ES
2022-01-03,0.005385,-0.005843,-0.004375,-0.005622,-0.005912
2022-01-04,0.004739,-0.000108,-0.001421,-0.000523,-0.000107
2022-01-05,-0.011618,-0.004358,-0.009122,-0.007709,-0.004270
2022-01-06,0.000357,-0.001053,-0.003379,-0.003720,-0.001053
2022-01-07,0.002029,-0.003006,-0.003329,-0.002646,-0.003001


In [14]:
print(
    "OOS portfolio return shape:",
    oos_portfolio_returns.shape
)

print()

print(
    "Missing values:"
)

print(
    oos_portfolio_returns
    .isna()
    .sum()
)

assert (
    oos_portfolio_returns
    .isna()
    .sum()
    .sum()
    == 0
)

assert (
    oos_portfolio_returns.shape[1]
    == 5
)

print(
    "\nOOS portfolio returns successfully constructed."
)

OOS portfolio return shape: (1203, 5)

Missing values:
Equal Weight Benchmark    0
Best GMVP                 0
Best Maximum Sharpe       0
Best Risk Parity          0
Best Minimum ES           0
dtype: int64

OOS portfolio returns successfully constructed.


In [15]:
oos_growth = (
    1
    +
    oos_portfolio_returns
).cumprod()

In [16]:
plt.figure(
    figsize=(11, 6)
)

for column in (
    oos_growth.columns
):

    plt.plot(
        oos_growth.index,
        oos_growth[column],
        label=column
    )

plt.xlabel(
    "Date"
)

plt.ylabel(
    "Growth of $1"
)

plt.title(
    "Out-of-Sample Portfolio Growth — 2022 Onward"
)

plt.grid(
    True,
    alpha=0.3
)

plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR
    / "oos_portfolio_growth.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

C:\Users\Jani\AppData\Local\Temp\ipykernel_30136\780192338.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
rf_raw = pd.read_csv(
    RF_FILE
)

rf_raw.columns = [
    str(col).strip()
    for col in rf_raw.columns
]

date_candidates = [
    "DATE",
    "Date",
    "date",
    "observation_date"
]

date_column = next(
    (
        col
        for col in date_candidates
        if col in rf_raw.columns
    ),
    None
)

if date_column is None:
    raise ValueError(
        "Risk-free date column not found."
    )

rf_raw[
    date_column
] = pd.to_datetime(
    rf_raw[
        date_column
    ],
    errors="coerce"
)

rf_raw[
    "DGS3MO"
] = pd.to_numeric(
    rf_raw[
        "DGS3MO"
    ],
    errors="coerce"
)

rf_clean = (
    rf_raw[
        [
            date_column,
            "DGS3MO"
        ]
    ]
    .dropna()
    .sort_values(
        date_column
    )
)

rf_annual = (
    rf_clean
    .set_index(
        date_column
    )[
        "DGS3MO"
    ]
    / 100
)

rf_daily_raw = (
    1
    +
    rf_annual
) ** (
    1 / 252
) - 1

In [18]:
combined_index = (
    rf_daily_raw.index
    .union(
        returns_oos.index
    )
)

rf_daily_oos = (
    rf_daily_raw
    .reindex(
        combined_index
    )
    .sort_index()
    .ffill()
    .reindex(
        returns_oos.index
    )
)

print(
    "OOS risk-free observations:",
    len(rf_daily_oos)
)

print(
    "Missing risk-free values:",
    rf_daily_oos
    .isna()
    .sum()
)

print(
    "Average annualized OOS risk-free rate:",
    f"{((1 + rf_daily_oos.mean()) ** 252 - 1):.4%}"
)

OOS risk-free observations: 1203
Missing risk-free values: 0
Average annualized OOS risk-free rate: 4.1197%


In [19]:
TRADING_DAYS = 252


def calculate_cagr(
    portfolio_returns
):

    total_growth = (
        1
        +
        portfolio_returns
    ).prod()

    years = (
        len(
            portfolio_returns
        )
        /
        TRADING_DAYS
    )

    return (
        total_growth
        ** (
            1 / years
        )
        - 1
    )

In [20]:
def calculate_drawdown(
    portfolio_returns
):

    wealth = (
        1
        +
        portfolio_returns
    ).cumprod()

    running_peak = (
        wealth.cummax()
    )

    drawdown = (
        wealth
        /
        running_peak
        - 1
    )

    return drawdown

In [21]:
def calculate_downside_deviation(
    excess_returns
):

    downside = np.minimum(
        excess_returns,
        0
    )

    return np.sqrt(
        np.mean(
            downside ** 2
        )
        * TRADING_DAYS
    )

In [22]:
oos_performance_rows = []

for portfolio in (
    oos_portfolio_returns.columns
):

    returns = (
        oos_portfolio_returns[
            portfolio
        ]
    )

    excess_returns = (
        returns
        -
        rf_daily_oos
    )

    annualized_return = (
        returns.mean()
        *
        TRADING_DAYS
    )

    cagr = (
        calculate_cagr(
            returns
        )
    )

    annualized_volatility = (
        returns.std(
            ddof=1
        )
        *
        np.sqrt(
            TRADING_DAYS
        )
    )

    annualized_excess_return = (
        excess_returns.mean()
        *
        TRADING_DAYS
    )

    sharpe_ratio = (
        annualized_excess_return
        /
        annualized_volatility
    )

    downside_deviation = (
        calculate_downside_deviation(
            excess_returns
        )
    )

    sortino_ratio = (
        annualized_excess_return
        /
        downside_deviation
    )

    drawdown = (
        calculate_drawdown(
            returns
        )
    )

    maximum_drawdown = (
        drawdown.min()
    )

    calmar_ratio = (
        cagr
        /
        abs(
            maximum_drawdown
        )
        if maximum_drawdown < 0
        else np.nan
    )

    terminal_wealth = (
        (
            1
            +
            returns
        )
        .prod()
    )

    oos_performance_rows.append({
        "Portfolio":
            portfolio,

        "Annualized Return":
            annualized_return,

        "CAGR":
            cagr,

        "Volatility":
            annualized_volatility,

        "Sharpe Ratio":
            sharpe_ratio,

        "Sortino Ratio":
            sortino_ratio,

        "Maximum Drawdown":
            maximum_drawdown,

        "Calmar Ratio":
            calmar_ratio,

        "Terminal Wealth":
            terminal_wealth
    })


oos_performance = pd.DataFrame(
    oos_performance_rows
).set_index(
    "Portfolio"
)

oos_performance

,Annualized Return,CAGR,Volatility,Sharpe Ratio,Sortino Ratio,Maximum Drawdown,Calmar Ratio,Terminal Wealth
Portfolio,,,,,,,,
Equal Weight Benchmark,0.137599,0.137391,0.132982,0.731109,1.050418,-0.173676,0.791077,1.848855
Best GMVP,0.007878,0.005984,0.061873,-0.525208,-0.721078,-0.164387,0.036399,1.028889
Best Maximum Sharpe,0.066331,0.064616,0.086208,0.301092,0.434318,-0.181008,0.356976,1.348383
Best Risk Parity,0.065134,0.063723,0.081931,0.302203,0.433203,-0.171095,0.372442,1.342994
Best Minimum ES,0.007167,0.005273,0.061801,-0.537322,-0.737413,-0.164101,0.032133,1.025424


In [23]:
oos_performance_display = (
    oos_performance.copy()
)

percentage_columns = [
    "Annualized Return",
    "CAGR",
    "Volatility",
    "Maximum Drawdown"
]

oos_performance_display[
    percentage_columns
] = (
    oos_performance_display[
        percentage_columns
    ]
    * 100
)

oos_performance_display.round(3)

,Annualized Return,CAGR,Volatility,Sharpe Ratio,Sortino Ratio,Maximum Drawdown,Calmar Ratio,Terminal Wealth
Portfolio,,,,,,,,
Equal Weight Benchmark,13.760,13.739,13.298,0.731,1.050,-17.368,0.791,1.849
Best GMVP,0.788,0.598,6.187,-0.525,-0.721,-16.439,0.036,1.029
Best Maximum Sharpe,6.633,6.462,8.621,0.301,0.434,-18.101,0.357,1.348
Best Risk Parity,6.513,6.372,8.193,0.302,0.433,-17.109,0.372,1.343
Best Minimum ES,0.717,0.527,6.180,-0.537,-0.737,-16.410,0.032,1.025


In [24]:
oos_rankings = pd.DataFrame(
    index=oos_performance.index
)

oos_rankings[
    "Return Rank"
] = (
    oos_performance[
        "CAGR"
    ]
    .rank(
        ascending=False,
        method="min"
    )
)

oos_rankings[
    "Volatility Rank"
] = (
    oos_performance[
        "Volatility"
    ]
    .rank(
        ascending=True,
        method="min"
    )
)

oos_rankings[
    "Sharpe Rank"
] = (
    oos_performance[
        "Sharpe Ratio"
    ]
    .rank(
        ascending=False,
        method="min"
    )
)

oos_rankings[
    "Sortino Rank"
] = (
    oos_performance[
        "Sortino Ratio"
    ]
    .rank(
        ascending=False,
        method="min"
    )
)

oos_rankings[
    "Drawdown Rank"
] = (
    oos_performance[
        "Maximum Drawdown"
    ]
    .rank(
        ascending=False,
        method="min"
    )
)

oos_rankings[
    "Calmar Rank"
] = (
    oos_performance[
        "Calmar Ratio"
    ]
    .rank(
        ascending=False,
        method="min"
    )
)

oos_rankings

,Return Rank,Volatility Rank,Sharpe Rank,Sortino Rank,Drawdown Rank,Calmar Rank
Portfolio,,,,,,
Equal Weight Benchmark,1.0,5.0,1.0,1.0,4.0,1.0
Best GMVP,4.0,2.0,4.0,4.0,2.0,4.0
Best Maximum Sharpe,2.0,4.0,3.0,2.0,5.0,3.0
Best Risk Parity,3.0,3.0,2.0,3.0,3.0,2.0
Best Minimum ES,5.0,1.0,5.0,5.0,1.0,5.0


In [25]:
oos_drawdowns = pd.DataFrame(
    index=oos_portfolio_returns.index
)

for portfolio in (
    oos_portfolio_returns.columns
):

    oos_drawdowns[
        portfolio
    ] = calculate_drawdown(
        oos_portfolio_returns[
            portfolio
        ]
    )

oos_drawdowns.head()

,Equal Weight Benchmark,Best GMVP,Best Maximum Sharpe,Best Risk Parity,Best Minimum ES
2022-01-03,0.000000,0.000000,0.000000,0.000000,0.000000
2022-01-04,0.000000,-0.000108,-0.001421,-0.000523,-0.000107
2022-01-05,-0.011618,-0.004465,-0.010530,-0.008228,-0.004376
2022-01-06,-0.011265,-0.005513,-0.013873,-0.011917,-0.005425
2022-01-07,-0.009259,-0.008503,-0.017156,-0.014531,-0.008410


In [26]:
plt.figure(
    figsize=(11, 6)
)

for portfolio in (
    oos_drawdowns.columns
):

    plt.plot(
        oos_drawdowns.index,
        oos_drawdowns[
            portfolio
        ]
        * 100,
        label=portfolio
    )

plt.xlabel(
    "Date"
)

plt.ylabel(
    "Drawdown (%)"
)

plt.title(
    "Out-of-Sample Portfolio Drawdowns — 2022 Onward"
)

plt.grid(
    True,
    alpha=0.3
)

plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR
    / "oos_portfolio_drawdowns.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

C:\Users\Jani\AppData\Local\Temp\ipykernel_30136\360970801.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [27]:
plt.figure(
    figsize=(9, 6)
)

for portfolio in (
    oos_performance.index
):

    x = (
        oos_performance.loc[
            portfolio,
            "Volatility"
        ]
        * 100
    )

    y = (
        oos_performance.loc[
            portfolio,
            "CAGR"
        ]
        * 100
    )

    plt.scatter(
        x,
        y,
        s=80
    )

    plt.annotate(
        portfolio,
        (
            x,
            y
        ),
        xytext=(5, 5),
        textcoords="offset points"
    )

plt.xlabel(
    "Annualized Volatility (%)"
)

plt.ylabel(
    "CAGR (%)"
)

plt.title(
    "Out-of-Sample Risk–Return Map"
)

plt.grid(
    True,
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR
    / "oos_risk_return_map.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

C:\Users\Jani\AppData\Local\Temp\ipykernel_30136\468696363.py:67: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [28]:
VAR_WINDOW = 500

VAR_CONFIDENCES = [
    0.95,
    0.99
]

print(
    "VaR estimation window:",
    VAR_WINDOW
)

print(
    "Confidence levels:",
    VAR_CONFIDENCES
)

VaR estimation window: 500
Confidence levels: [0.95, 0.99]


In [29]:
full_portfolio_returns = pd.DataFrame(
    index=simple_returns.index
)

for _, row in shortlist.iterrows():

    assets = [
        x.strip()
        for x in row[
            "Subset"
        ].split(",")
    ]

    weights = row[
        "Frozen Weights"
    ]

    label = row[
        "Portfolio Label"
    ]

    full_portfolio_returns[
        label
    ] = portfolio_returns_from_weights(
        assets,
        weights,
        simple_returns
    )


print(
    "Full frozen portfolio return shape:",
    full_portfolio_returns.shape
)

full_portfolio_returns.head()

Full frozen portfolio return shape: (5512, 5)


,Equal Weight Benchmark,Best GMVP,Best Maximum Sharpe,Best Risk Parity,Best Minimum ES
2004-11-19,-0.004060,-0.002758,-0.003547,-0.002326,-0.002709
2004-11-22,0.004905,0.001814,0.001821,0.001937,0.001796
2004-11-23,0.000303,-0.000418,-0.001891,-0.001891,-0.000430
2004-11-24,0.005955,0.001084,0.003275,0.003190,0.001077
2004-11-26,0.003844,-0.002750,-0.001527,-0.001202,-0.002762


In [30]:
def historical_var(
    returns,
    confidence
):

    return -np.quantile(
        returns,
        1 - confidence
    )

In [31]:
def parametric_var(
    returns,
    confidence
):

    mu = returns.mean()

    sigma = returns.std(
        ddof=1
    )

    z = norm.ppf(
        1 - confidence
    )

    return -(
        mu
        +
        z * sigma
    )

In [32]:
def rolling_var_backtest(
    return_series,
    oos_dates,
    window=500,
    confidence=0.99
):

    rows = []

    for date in oos_dates:

        location = (
            return_series.index
            .get_loc(date)
        )

        estimation_sample = (
            return_series
            .iloc[
                location - window:
                location
            ]
        )

        if (
            len(estimation_sample)
            < window
        ):
            continue

        historical_forecast = (
            historical_var(
                estimation_sample,
                confidence
            )
        )

        parametric_forecast = (
            parametric_var(
                estimation_sample,
                confidence
            )
        )

        realized_return = (
            return_series.loc[
                date
            ]
        )

        realized_loss = (
            -realized_return
        )

        rows.append({

            "Date":
                date,

            "Confidence":
                confidence,

            "Realized Return":
                realized_return,

            "Realized Loss":
                realized_loss,

            "Historical VaR":
                historical_forecast,

            "Parametric VaR":
                parametric_forecast,

            "Historical Exceedance":
                realized_loss
                >
                historical_forecast,

            "Parametric Exceedance":
                realized_loss
                >
                parametric_forecast
        })

    return (
        pd.DataFrame(
            rows
        )
        .set_index(
            "Date"
        )
    )

In [33]:
test_portfolio = (
    "Equal Weight Benchmark"
)

test_backtest = (
    rolling_var_backtest(
        full_portfolio_returns[
            test_portfolio
        ],
        returns_oos.index,
        window=VAR_WINDOW,
        confidence=0.99
    )
)

print(
    "Backtest observations:",
    len(test_backtest)
)

test_backtest.head()

Backtest observations: 1203


,Confidence,Realized Return,Realized Loss,Historical VaR,Parametric VaR,Historical Exceedance,Parametric Exceedance
Date,,,,,,,
2022-01-03,0.99,0.005385,-0.005385,0.037833,0.029663,False,False
2022-01-04,0.99,0.004739,-0.004739,0.037833,0.029662,False,False
2022-01-05,0.99,-0.011618,0.011618,0.037833,0.029655,False,False
2022-01-06,0.99,0.000357,-0.000357,0.037833,0.029711,False,False
2022-01-07,0.99,0.002029,-0.002029,0.037833,0.029707,False,False


In [34]:
print(
    "First forecast date:",
    test_backtest.index.min()
)

print(
    "Last forecast date:",
    test_backtest.index.max()
)

print()

print(
    "Historical exceedances:",
    test_backtest[
        "Historical Exceedance"
    ].sum()
)

print(
    "Parametric exceedances:",
    test_backtest[
        "Parametric Exceedance"
    ].sum()
)

First forecast date: 2022-01-03 00:00:00
Last forecast date: 2026-10-19 00:00:00

Historical exceedances: 17
Parametric exceedances: 22


In [35]:
backtest_rows = []

for portfolio in (
    full_portfolio_returns.columns
):

    print(
        "Backtesting:",
        portfolio
    )

    series = (
        full_portfolio_returns[
            portfolio
        ]
    )

    for confidence in (
        VAR_CONFIDENCES
    ):

        result = (
            rolling_var_backtest(
                series,
                returns_oos.index,
                window=VAR_WINDOW,
                confidence=confidence
            )
            .reset_index()
        )

        result[
            "Portfolio"
        ] = portfolio

        backtest_rows.append(
            result
        )


var_backtests = pd.concat(
    backtest_rows,
    ignore_index=True
)

print(
    "\nTotal backtesting rows:",
    len(var_backtests)
)

var_backtests.head()

Backtesting: Equal Weight Benchmark


Backtesting: Best GMVP


Backtesting: Best Maximum Sharpe


Backtesting: Best Risk Parity


Backtesting: Best Minimum ES



Total backtesting rows: 12030


,Date,Confidence,Realized Return,Realized Loss,Historical VaR,Parametric VaR,Historical Exceedance,Parametric Exceedance,Portfolio
0,2022-01-03,0.95,0.005385,-0.005385,0.0163,0.020793,False,False,Equal Weight Benchmark
1,2022-01-04,0.95,0.004739,-0.004739,0.0163,0.020792,False,False,Equal Weight Benchmark
2,2022-01-05,0.95,-0.011618,0.011618,0.0163,0.020784,False,False,Equal Weight Benchmark
3,2022-01-06,0.95,0.000357,-0.000357,0.0163,0.020834,False,False,Equal Weight Benchmark
4,2022-01-07,0.95,0.002029,-0.002029,0.0163,0.020829,False,False,Equal Weight Benchmark


In [36]:
exceedance_summary_rows = []

for (
    portfolio,
    confidence
), group in (
    var_backtests
    .groupby(
        [
            "Portfolio",
            "Confidence"
        ]
    )
):

    n = len(group)

    expected_rate = (
        1
        -
        confidence
    )

    historical_n = (
        group[
            "Historical Exceedance"
        ]
        .sum()
    )

    parametric_n = (
        group[
            "Parametric Exceedance"
        ]
        .sum()
    )

    exceedance_summary_rows.append({

        "Portfolio":
            portfolio,

        "Confidence":
            confidence,

        "Observations":
            n,

        "Expected Exceedance Rate":
            expected_rate,

        "Expected Exceedances":
            n * expected_rate,

        "Historical Exceedances":
            historical_n,

        "Historical Rate":
            historical_n / n,

        "Parametric Exceedances":
            parametric_n,

        "Parametric Rate":
            parametric_n / n
    })


exceedance_summary = (
    pd.DataFrame(
        exceedance_summary_rows
    )
)

exceedance_summary

,Portfolio,Confidence,Observations,Expected Exceedance Rate,Expected Exceedances,Historical Exceedances,Historical Rate,Parametric Exceedances,Parametric Rate
0,Best GMVP,0.95,1203,0.05,60.15,76,0.063175,69,0.057357
1,Best GMVP,0.99,1203,0.01,12.03,15,0.012469,26,0.021613
2,Best Maximum Sharpe,0.95,1203,0.05,60.15,75,0.062344,77,0.064007
3,Best Maximum Sharpe,0.99,1203,0.01,12.03,15,0.012469,20,0.016625
4,Best Minimum ES,0.95,1203,0.05,60.15,76,0.063175,68,0.056525
5,Best Minimum ES,0.99,1203,0.01,12.03,15,0.012469,27,0.022444
6,Best Risk Parity,0.95,1203,0.05,60.15,66,0.054863,71,0.059019
7,Best Risk Parity,0.99,1203,0.01,12.03,15,0.012469,26,0.021613
8,Equal Weight Benchmark,0.95,1203,0.05,60.15,66,0.054863,55,0.045719
9,Equal Weight Benchmark,0.99,1203,0.01,12.03,17,0.014131,22,0.018288


In [37]:
exceedance_display = (
    exceedance_summary.copy()
)

exceedance_display[
    "Expected Exceedance Rate"
] *= 100

exceedance_display[
    "Historical Rate"
] *= 100

exceedance_display[
    "Parametric Rate"
] *= 100

exceedance_display.round(
    3
)

,Portfolio,Confidence,Observations,Expected Exceedance Rate,Expected Exceedances,Historical Exceedances,Historical Rate,Parametric Exceedances,Parametric Rate
0,Best GMVP,0.95,1203,5.0,60.15,76,6.318,69,5.736
1,Best GMVP,0.99,1203,1.0,12.03,15,1.247,26,2.161
2,Best Maximum Sharpe,0.95,1203,5.0,60.15,75,6.234,77,6.401
3,Best Maximum Sharpe,0.99,1203,1.0,12.03,15,1.247,20,1.663
4,Best Minimum ES,0.95,1203,5.0,60.15,76,6.318,68,5.653
5,Best Minimum ES,0.99,1203,1.0,12.03,15,1.247,27,2.244
6,Best Risk Parity,0.95,1203,5.0,60.15,66,5.486,71,5.902
7,Best Risk Parity,0.99,1203,1.0,12.03,15,1.247,26,2.161
8,Equal Weight Benchmark,0.95,1203,5.0,60.15,66,5.486,55,4.572
9,Equal Weight Benchmark,0.99,1203,1.0,12.03,17,1.413,22,1.829


In [38]:
def kupiec_test(
    exceedances,
    confidence
):

    exceedances = np.asarray(
        exceedances,
        dtype=int
    )

    T = len(exceedances)

    x = exceedances.sum()

    p = 1 - confidence

    p_hat = x / T

    # Avoid log(0)
    eps = 1e-12

    p_hat_safe = np.clip(
        p_hat,
        eps,
        1 - eps
    )

    p_safe = np.clip(
        p,
        eps,
        1 - eps
    )

    log_likelihood_null = (
        (T - x)
        * np.log(
            1 - p_safe
        )
        +
        x
        * np.log(
            p_safe
        )
    )

    log_likelihood_alt = (
        (T - x)
        * np.log(
            1 - p_hat_safe
        )
        +
        x
        * np.log(
            p_hat_safe
        )
    )

    lr_uc = (
        -2
        * (
            log_likelihood_null
            -
            log_likelihood_alt
        )
    )

    p_value = (
        1
        -
        chi2.cdf(
            lr_uc,
            df=1
        )
    )

    return {

        "Observations":
            T,

        "Exceedances":
            int(x),

        "Expected Rate":
            p,

        "Observed Rate":
            p_hat,

        "Kupiec LR":
            lr_uc,

        "Kupiec p-value":
            p_value,

        "Kupiec Pass":
            p_value >= 0.05
    }

In [39]:
ew_99 = (
    var_backtests[
        (
            var_backtests[
                "Portfolio"
            ]
            ==
            "Equal Weight Benchmark"
        )
        &
        (
            var_backtests[
                "Confidence"
            ]
            ==
            0.99
        )
    ]
)

ew_hist_kupiec = (
    kupiec_test(
        ew_99[
            "Historical Exceedance"
        ],
        confidence=0.99
    )
)

ew_hist_kupiec

{'Observations': 1203,
 'Exceedances': 17,
 'Expected Rate': 0.010000000000000009,
 'Observed Rate': np.float64(0.014131338320864505),
 'Kupiec LR': np.float64(1.8383027417294215),
 'Kupiec p-value': np.float64(0.1751500066316336),
 'Kupiec Pass': np.True_}

In [40]:
kupiec_rows = []

for (
    portfolio,
    confidence
), group in (
    var_backtests
    .groupby(
        [
            "Portfolio",
            "Confidence"
        ]
    )
):

    # Historical VaR
    hist_result = (
        kupiec_test(
            group[
                "Historical Exceedance"
            ],
            confidence
        )
    )

    kupiec_rows.append({

        "Portfolio":
            portfolio,

        "Model":
            "Historical",

        "Confidence":
            confidence,

        **hist_result
    })

    # Parametric Normal VaR
    param_result = (
        kupiec_test(
            group[
                "Parametric Exceedance"
            ],
            confidence
        )
    )

    kupiec_rows.append({

        "Portfolio":
            portfolio,

        "Model":
            "Parametric Normal",

        "Confidence":
            confidence,

        **param_result
    })


kupiec_results = (
    pd.DataFrame(
        kupiec_rows
    )
)

kupiec_results

,Portfolio,Model,Confidence,Observations,Exceedances,Expected Rate,Observed Rate,Kupiec LR,Kupiec p-value,Kupiec Pass
0,Best GMVP,Historical,0.95,1203,76,0.05,0.063175,4.072413,0.043589,False
1,Best GMVP,Parametric Normal,0.95,1203,69,0.05,0.057357,1.311289,0.252161,True
2,Best GMVP,Historical,0.99,1203,15,0.01,0.012469,0.686813,0.407251,True
3,Best GMVP,Parametric Normal,0.99,1203,26,0.01,0.021613,12.300548,0.000453,False
4,Best Maximum Sharpe,Historical,0.95,1203,75,0.05,0.062344,3.590800,0.058100,True
5,Best Maximum Sharpe,Parametric Normal,0.95,1203,77,0.05,0.064007,4.582117,0.032307,False
6,Best Maximum Sharpe,Historical,0.99,1203,15,0.01,0.012469,0.686813,0.407251,True
7,Best Maximum Sharpe,Parametric Normal,0.99,1203,20,0.01,0.016625,4.446605,0.034971,False
8,Best Minimum ES,Historical,0.95,1203,76,0.05,0.063175,4.072413,0.043589,False
9,Best Minimum ES,Parametric Normal,0.95,1203,68,0.05,0.056525,1.036656,0.308601,True


In [41]:
print(
    "Kupiec result rows:",
    len(kupiec_results)
)

print()

print(
    kupiec_results[
        "Kupiec Pass"
    ].value_counts()
)

Kupiec result rows: 20

Kupiec Pass
True     12
False     8
Name: count, dtype: int64


In [42]:
kupiec_display = (
    kupiec_results.copy()
)

kupiec_display[
    "Expected Rate"
] *= 100

kupiec_display[
    "Observed Rate"
] *= 100

kupiec_display[
    [
        "Expected Rate",
        "Observed Rate",
        "Kupiec LR",
        "Kupiec p-value"
    ]
].round(4)

,Expected Rate,Observed Rate,Kupiec LR,Kupiec p-value
0,5.0,6.3175,4.0724,0.0436
1,5.0,5.7357,1.3113,0.2522
2,1.0,1.2469,0.6868,0.4073
3,1.0,2.1613,12.3005,0.0005
4,5.0,6.2344,3.5908,0.0581
5,5.0,6.4007,4.5821,0.0323
6,1.0,1.2469,0.6868,0.4073
7,1.0,1.6625,4.4466,0.0350
8,5.0,6.3175,4.0724,0.0436
9,5.0,5.6525,1.0367,0.3086


In [43]:
kupiec_display[
    [
        "Portfolio",
        "Model",
        "Confidence",
        "Observations",
        "Exceedances",
        "Expected Rate",
        "Observed Rate",
        "Kupiec LR",
        "Kupiec p-value",
        "Kupiec Pass"
    ]
].round(4)

,Portfolio,Model,Confidence,Observations,Exceedances,Expected Rate,Observed Rate,Kupiec LR,Kupiec p-value,Kupiec Pass
0,Best GMVP,Historical,0.95,1203,76,5.0,6.3175,4.0724,0.0436,False
1,Best GMVP,Parametric Normal,0.95,1203,69,5.0,5.7357,1.3113,0.2522,True
2,Best GMVP,Historical,0.99,1203,15,1.0,1.2469,0.6868,0.4073,True
3,Best GMVP,Parametric Normal,0.99,1203,26,1.0,2.1613,12.3005,0.0005,False
4,Best Maximum Sharpe,Historical,0.95,1203,75,5.0,6.2344,3.5908,0.0581,True
5,Best Maximum Sharpe,Parametric Normal,0.95,1203,77,5.0,6.4007,4.5821,0.0323,False
6,Best Maximum Sharpe,Historical,0.99,1203,15,1.0,1.2469,0.6868,0.4073,True
7,Best Maximum Sharpe,Parametric Normal,0.99,1203,20,1.0,1.6625,4.4466,0.0350,False
8,Best Minimum ES,Historical,0.95,1203,76,5.0,6.3175,4.0724,0.0436,False
9,Best Minimum ES,Parametric Normal,0.95,1203,68,5.0,5.6525,1.0367,0.3086,True


In [44]:
def christoffersen_independence_test(
    exceedances
):

    x = np.asarray(
        exceedances,
        dtype=int
    )

    previous = x[:-1]
    current = x[1:]

    n00 = np.sum(
        (previous == 0)
        &
        (current == 0)
    )

    n01 = np.sum(
        (previous == 0)
        &
        (current == 1)
    )

    n10 = np.sum(
        (previous == 1)
        &
        (current == 0)
    )

    n11 = np.sum(
        (previous == 1)
        &
        (current == 1)
    )

    eps = 1e-12

    pi01 = (
        n01
        /
        max(
            n00 + n01,
            1
        )
    )

    pi11 = (
        n11
        /
        max(
            n10 + n11,
            1
        )
    )

    pi = (
        (n01 + n11)
        /
        max(
            n00 + n01 + n10 + n11,
            1
        )
    )

    pi01 = np.clip(
        pi01,
        eps,
        1 - eps
    )

    pi11 = np.clip(
        pi11,
        eps,
        1 - eps
    )

    pi = np.clip(
        pi,
        eps,
        1 - eps
    )

    log_likelihood_null = (
        (n00 + n10)
        * np.log(
            1 - pi
        )
        +
        (n01 + n11)
        * np.log(
            pi
        )
    )

    log_likelihood_alt = (
        n00
        * np.log(
            1 - pi01
        )
        +
        n01
        * np.log(
            pi01
        )
        +
        n10
        * np.log(
            1 - pi11
        )
        +
        n11
        * np.log(
            pi11
        )
    )

    lr_ind = (
        -2
        * (
            log_likelihood_null
            -
            log_likelihood_alt
        )
    )

    p_value = (
        1
        -
        chi2.cdf(
            lr_ind,
            df=1
        )
    )

    return {

        "N00": int(n00),
        "N01": int(n01),
        "N10": int(n10),
        "N11": int(n11),

        "Pi01":
            pi01,

        "Pi11":
            pi11,

        "Christoffersen LR":
            lr_ind,

        "Christoffersen p-value":
            p_value,

        "Independence Pass":
            p_value >= 0.05
    }

In [45]:
ew_hist_independence = (
    christoffersen_independence_test(
        ew_99[
            "Historical Exceedance"
        ]
    )
)

ew_hist_independence

{'N00': 1169,
 'N01': 16,
 'N10': 16,
 'N11': 1,
 'Pi01': np.float64(0.01350210970464135),
 'Pi11': np.float64(0.058823529411764705),
 'Christoffersen LR': np.float64(1.401903519809082),
 'Christoffersen p-value': np.float64(0.2364051192606531),
 'Independence Pass': np.True_}

In [46]:
christoffersen_rows = []

for (
    portfolio,
    confidence
), group in (
    var_backtests
    .groupby(
        [
            "Portfolio",
            "Confidence"
        ]
    )
):

    # Historical
    hist_result = (
        christoffersen_independence_test(
            group[
                "Historical Exceedance"
            ]
        )
    )

    christoffersen_rows.append({

        "Portfolio":
            portfolio,

        "Model":
            "Historical",

        "Confidence":
            confidence,

        **hist_result
    })


    # Parametric Normal
    param_result = (
        christoffersen_independence_test(
            group[
                "Parametric Exceedance"
            ]
        )
    )

    christoffersen_rows.append({

        "Portfolio":
            portfolio,

        "Model":
            "Parametric Normal",

        "Confidence":
            confidence,

        **param_result
    })


christoffersen_results = (
    pd.DataFrame(
        christoffersen_rows
    )
)

christoffersen_results

,Portfolio,Model,Confidence,N00,N01,N10,N11,Pi01,Pi11,Christoffersen LR,Christoffersen p-value,Independence Pass
0,Best GMVP,Historical,0.95,1058,68,69,7,0.060391,9.210526e-02,1.086716,0.297200,True
1,Best GMVP,Parametric Normal,0.95,1071,62,62,7,0.054722,1.014493e-01,2.191135,0.138807,True
2,Best GMVP,Historical,0.99,1172,15,15,0,0.012637,1.000000e-12,0.379117,0.538076,True
3,Best GMVP,Parametric Normal,0.99,1151,25,25,1,0.021259,3.846154e-02,0.291200,0.589453,True
4,Best Maximum Sharpe,Historical,0.95,1058,69,69,6,0.061224,8.000000e-02,0.393246,0.530598,True
5,Best Maximum Sharpe,Parametric Normal,0.95,1055,70,70,7,0.062222,9.090909e-02,0.889626,0.345579,True
6,Best Maximum Sharpe,Historical,0.99,1173,14,14,1,0.011794,6.666667e-02,1.817102,0.177659,True
7,Best Maximum Sharpe,Parametric Normal,0.99,1164,18,18,2,0.015228,1.000000e-01,4.132657,0.042063,False
8,Best Minimum ES,Historical,0.95,1058,68,69,7,0.060391,9.210526e-02,1.086716,0.297200,True
9,Best Minimum ES,Parametric Normal,0.95,1072,62,62,6,0.054674,8.823529e-02,1.178174,0.277728,True


In [47]:
print(
    "Christoffersen result rows:",
    len(
        christoffersen_results
    )
)

print()

print(
    christoffersen_results[
        "Independence Pass"
    ].value_counts()
)

Christoffersen result rows: 20

Independence Pass
True     17
False     3
Name: count, dtype: int64


In [48]:
christoffersen_display = (
    christoffersen_results[
        [
            "Portfolio",
            "Model",
            "Confidence",
            "N00",
            "N01",
            "N10",
            "N11",
            "Pi01",
            "Pi11",
            "Christoffersen LR",
            "Christoffersen p-value",
            "Independence Pass"
        ]
    ]
    .copy()
)

christoffersen_display.round(4)

,Portfolio,Model,Confidence,N00,N01,N10,N11,Pi01,Pi11,Christoffersen LR,Christoffersen p-value,Independence Pass
0,Best GMVP,Historical,0.95,1058,68,69,7,0.0604,0.0921,1.0867,0.2972,True
1,Best GMVP,Parametric Normal,0.95,1071,62,62,7,0.0547,0.1014,2.1911,0.1388,True
2,Best GMVP,Historical,0.99,1172,15,15,0,0.0126,0.0000,0.3791,0.5381,True
3,Best GMVP,Parametric Normal,0.99,1151,25,25,1,0.0213,0.0385,0.2912,0.5895,True
4,Best Maximum Sharpe,Historical,0.95,1058,69,69,6,0.0612,0.0800,0.3932,0.5306,True
5,Best Maximum Sharpe,Parametric Normal,0.95,1055,70,70,7,0.0622,0.0909,0.8896,0.3456,True
6,Best Maximum Sharpe,Historical,0.99,1173,14,14,1,0.0118,0.0667,1.8171,0.1777,True
7,Best Maximum Sharpe,Parametric Normal,0.99,1164,18,18,2,0.0152,0.1000,4.1327,0.0421,False
8,Best Minimum ES,Historical,0.95,1058,68,69,7,0.0604,0.0921,1.0867,0.2972,True
9,Best Minimum ES,Parametric Normal,0.95,1072,62,62,6,0.0547,0.0882,1.1782,0.2777,True


In [49]:
conditional_coverage = (
    kupiec_results.merge(
        christoffersen_results,
        on=[
            "Portfolio",
            "Model",
            "Confidence"
        ],
        how="inner"
    )
)

print(
    "Merged rows:",
    len(
        conditional_coverage
    )
)

Merged rows: 20


In [50]:
conditional_coverage[
    "Conditional Coverage LR"
] = (
    conditional_coverage[
        "Kupiec LR"
    ]
    +
    conditional_coverage[
        "Christoffersen LR"
    ]
)

conditional_coverage[
    "Conditional Coverage p-value"
] = (
    1
    -
    chi2.cdf(
        conditional_coverage[
            "Conditional Coverage LR"
        ],
        df=2
    )
)

conditional_coverage[
    "Conditional Coverage Pass"
] = (
    conditional_coverage[
        "Conditional Coverage p-value"
    ]
    >= 0.05
)

In [51]:
coverage_summary = (
    conditional_coverage[
        [
            "Portfolio",
            "Model",
            "Confidence",
            "Observations",
            "Exceedances",
            "Expected Rate",
            "Observed Rate",
            "Kupiec LR",
            "Kupiec p-value",
            "Kupiec Pass",
            "Christoffersen LR",
            "Christoffersen p-value",
            "Independence Pass",
            "Conditional Coverage LR",
            "Conditional Coverage p-value",
            "Conditional Coverage Pass"
        ]
    ]
    .copy()
)

coverage_summary.round(4)

,Portfolio,Model,Confidence,Observations,Exceedances,Expected Rate,Observed Rate,Kupiec LR,Kupiec p-value,Kupiec Pass,Christoffersen LR,Christoffersen p-value,Independence Pass,Conditional Coverage LR,Conditional Coverage p-value,Conditional Coverage Pass
0,Best GMVP,Historical,0.95,1203,76,0.05,0.0632,4.0724,0.0436,False,1.0867,0.2972,True,5.1591,0.0758,True
1,Best GMVP,Parametric Normal,0.95,1203,69,0.05,0.0574,1.3113,0.2522,True,2.1911,0.1388,True,3.5024,0.1736,True
2,Best GMVP,Historical,0.99,1203,15,0.01,0.0125,0.6868,0.4073,True,0.3791,0.5381,True,1.0659,0.5869,True
3,Best GMVP,Parametric Normal,0.99,1203,26,0.01,0.0216,12.3005,0.0005,False,0.2912,0.5895,True,12.5917,0.0018,False
4,Best Maximum Sharpe,Historical,0.95,1203,75,0.05,0.0623,3.5908,0.0581,True,0.3932,0.5306,True,3.9840,0.1364,True
5,Best Maximum Sharpe,Parametric Normal,0.95,1203,77,0.05,0.0640,4.5821,0.0323,False,0.8896,0.3456,True,5.4717,0.0648,True
6,Best Maximum Sharpe,Historical,0.99,1203,15,0.01,0.0125,0.6868,0.4073,True,1.8171,0.1777,True,2.5039,0.2859,True
7,Best Maximum Sharpe,Parametric Normal,0.99,1203,20,0.01,0.0166,4.4466,0.0350,False,4.1327,0.0421,False,8.5793,0.0137,False
8,Best Minimum ES,Historical,0.95,1203,76,0.05,0.0632,4.0724,0.0436,False,1.0867,0.2972,True,5.1591,0.0758,True
9,Best Minimum ES,Parametric Normal,0.95,1203,68,0.05,0.0565,1.0367,0.3086,True,1.1782,0.2777,True,2.2148,0.3304,True


In [52]:
coverage_pass_summary = (
    coverage_summary
    .groupby(
        [
            "Model",
            "Confidence"
        ]
    )
    [
        [
            "Kupiec Pass",
            "Independence Pass",
            "Conditional Coverage Pass"
        ]
    ]
    .sum()
)

coverage_pass_summary

Kupiec Pass  Independence Pass  \
Model             Confidence                                   
Historical        0.95                  3                  4   
                  0.99                  5                  5   
Parametric Normal 0.95                  4                  4   
                  0.99                  0                  4   

                              Conditional Coverage Pass  
Model             Confidence                             
Historical        0.95                                5  
                  0.99                                5  
Parametric Normal 0.95                                4  
                  0.99                                0

In [53]:
BASEL_WINDOW = 250


def basel_zone(
    exceptions
):

    if exceptions <= 4:
        return "Green"

    elif exceptions <= 9:
        return "Yellow"

    else:
        return "Red"

In [54]:
basel_rows = []

for (
    portfolio,
    confidence
), group in (
    var_backtests
    .groupby(
        [
            "Portfolio",
            "Confidence"
        ]
    )
):

    # Basel traffic light is for 99% VaR
    if confidence != 0.99:
        continue

    group = (
        group
        .sort_values("Date")
        .copy()
        .reset_index(drop=True)
    )

    dates = pd.to_datetime(
        group["Date"]
    )


    # -------------------------
    # Historical VaR
    # -------------------------

    historical_rolling = (
        group[
            "Historical Exceedance"
        ]
        .astype(int)
        .rolling(
            window=BASEL_WINDOW
        )
        .sum()
    )


    for i in range(
        BASEL_WINDOW - 1,
        len(group)
    ):

        count = int(
            historical_rolling.iloc[i]
        )

        basel_rows.append({

            "Date":
                dates.iloc[i],

            "Portfolio":
                portfolio,

            "Model":
                "Historical",

            "Exceptions":
                count,

            "Zone":
                basel_zone(
                    count
                )
        })


    # -------------------------
    # Parametric Normal VaR
    # -------------------------

    parametric_rolling = (
        group[
            "Parametric Exceedance"
        ]
        .astype(int)
        .rolling(
            window=BASEL_WINDOW
        )
        .sum()
    )


    for i in range(
        BASEL_WINDOW - 1,
        len(group)
    ):

        count = int(
            parametric_rolling.iloc[i]
        )

        basel_rows.append({

            "Date":
                dates.iloc[i],

            "Portfolio":
                portfolio,

            "Model":
                "Parametric Normal",

            "Exceptions":
                count,

            "Zone":
                basel_zone(
                    count
                )
        })


basel_results = pd.DataFrame(
    basel_rows
)


print(
    "Basel rows:",
    len(basel_results)
)

print(
    "First Basel date:",
    basel_results["Date"].min()
)

print(
    "Last Basel date:",
    basel_results["Date"].max()
)

basel_results.head()

Basel rows: 9540
First Basel date: 2022-12-29 00:00:00
Last Basel date: 2026-10-19 00:00:00


,Date,Portfolio,Model,Exceptions,Zone
0,2022-12-29,Best GMVP,Historical,10,Red
1,2022-12-30,Best GMVP,Historical,10,Red
2,2023-01-03,Best GMVP,Historical,10,Red
3,2023-01-04,Best GMVP,Historical,10,Red
4,2023-01-05,Best GMVP,Historical,10,Red


In [55]:
latest_basel = (
    basel_results
    .sort_values("Date")
    .groupby(
        [
            "Portfolio",
            "Model"
        ]
    )
    .tail(1)
    [
        [
            "Date",
            "Portfolio",
            "Model",
            "Exceptions",
            "Zone"
        ]
    ]
    .sort_values(
        [
            "Model",
            "Portfolio"
        ]
    )
)

latest_basel

,Date,Portfolio,Model,Exceptions,Zone
953,2026-10-19,Best GMVP,Historical,2,Green
2861,2026-10-19,Best Maximum Sharpe,Historical,4,Green
4769,2026-10-19,Best Minimum ES,Historical,2,Green
6677,2026-10-19,Best Risk Parity,Historical,4,Green
8585,2026-10-19,Equal Weight Benchmark,Historical,3,Green
1907,2026-10-19,Best GMVP,Parametric Normal,4,Green
3815,2026-10-19,Best Maximum Sharpe,Parametric Normal,5,Yellow
5723,2026-10-19,Best Minimum ES,Parametric Normal,3,Green
7631,2026-10-19,Best Risk Parity,Parametric Normal,11,Red
9539,2026-10-19,Equal Weight Benchmark,Parametric Normal,6,Yellow


In [56]:
basel_zone_summary = (
    basel_results
    .groupby(
        [
            "Portfolio",
            "Model",
            "Zone"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

basel_zone_summary

Zone                                      Green  Red  Yellow
Portfolio              Model                                
Best GMVP              Historical           842   28      84
                       Parametric Normal    784   86      84
Best Maximum Sharpe    Historical           670    0     284
                       Parametric Normal    547   49     358
Best Minimum ES        Historical           842   28      84
                       Parametric Normal    779  111      64
Best Risk Parity       Historical           862    0      92
                       Parametric Normal    571  135     248
Equal Weight Benchmark Historical           609    0     345
                       Parametric Normal    454    0     500

In [57]:
basel_zone_percent = (
    basel_zone_summary
    .div(
        basel_zone_summary.sum(
            axis=1
        ),
        axis=0
    )
    * 100
)

basel_zone_percent.round(2)

Zone                                      Green    Red  Yellow
Portfolio              Model                                  
Best GMVP              Historical         88.26   2.94    8.81
                       Parametric Normal  82.18   9.01    8.81
Best Maximum Sharpe    Historical         70.23   0.00   29.77
                       Parametric Normal  57.34   5.14   37.53
Best Minimum ES        Historical         88.26   2.94    8.81
                       Parametric Normal  81.66  11.64    6.71
Best Risk Parity       Historical         90.36   0.00    9.64
                       Parametric Normal  59.85  14.15   26.00
Equal Weight Benchmark Historical         63.84   0.00   36.16
                       Parametric Normal  47.59   0.00   52.41

In [58]:
basel_max_exceptions = (
    basel_results
    .groupby(
        [
            "Portfolio",
            "Model"
        ]
    )
    ["Exceptions"]
    .agg(
        [
            "min",
            "mean",
            "max"
        ]
    )
)

basel_max_exceptions.round(2)

min  mean  max
Portfolio              Model                            
Best GMVP              Historical           0  2.07   10
                       Parametric Normal    0  3.58   17
Best Maximum Sharpe    Historical           0  2.63    6
                       Parametric Normal    0  3.18   10
Best Minimum ES        Historical           0  2.07   10
                       Parametric Normal    0  3.79   19
Best Risk Parity       Historical           0  2.62    7
                       Parametric Normal    0  4.03   11
Equal Weight Benchmark Historical           0  3.19    7
                       Parametric Normal    0  4.01    8

In [59]:
plt.figure(
    figsize=(12, 7)
)


for (
    portfolio,
    model
), group in (
    basel_results
    .groupby(
        [
            "Portfolio",
            "Model"
        ]
    )
):

    plt.plot(
        group["Date"],
        group["Exceptions"],
        label=(
            portfolio
            +
            " | "
            +
            model
        ),
        alpha=0.75
    )


plt.axhline(
    y=4,
    linestyle="--",
    label="Green upper limit"
)

plt.axhline(
    y=9,
    linestyle="--",
    label="Yellow upper limit"
)


plt.xlabel(
    "Date"
)

plt.ylabel(
    "Exceptions in Previous 250 Trading Days"
)

plt.title(
    "Rolling Basel Traffic-Light Backtest — 99% VaR"
)

plt.legend(
    bbox_to_anchor=(
        1.02,
        1
    ),
    loc="upper left"
)

plt.grid(
    True,
    alpha=0.3
)

plt.tight_layout()


plt.savefig(
    FIGURE_DIR
    /
    "basel_traffic_light_rolling.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

C:\Users\Jani\AppData\Local\Temp\ipykernel_30136\3227589566.py:82: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [60]:
basel_model_average = (
    basel_results
    .groupby(
        [
            "Date",
            "Model"
        ]
    )
    ["Exceptions"]
    .mean()
    .unstack()
)

basel_model_average.head()

Model,Historical,Parametric Normal
Date,,
2022-12-29,7.8,12.6
2022-12-30,7.8,12.6
2023-01-03,7.8,12.6
2023-01-04,7.8,12.6
2023-01-05,7.8,12.6


In [61]:
plt.figure(
    figsize=(11, 6)
)


for model in (
    basel_model_average.columns
):

    plt.plot(
        basel_model_average.index,
        basel_model_average[
            model
        ],
        label=model
    )


plt.axhline(
    4,
    linestyle="--",
    label="Green / Yellow Boundary"
)

plt.axhline(
    9,
    linestyle="--",
    label="Yellow / Red Boundary"
)


plt.xlabel(
    "Date"
)

plt.ylabel(
    "Average 250-Day VaR Exceptions"
)

plt.title(
    "Historical vs Parametric Normal — Basel Exception Experience"
)

plt.legend()

plt.grid(
    True,
    alpha=0.3
)

plt.tight_layout()


plt.savefig(
    FIGURE_DIR
    /
    "basel_model_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

C:\Users\Jani\AppData\Local\Temp\ipykernel_30136\1762295780.py:62: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [62]:
# Create model-validation scorecard

validation_rows = []

for model in [
    "Historical",
    "Parametric Normal"
]:

    for confidence in [
        0.95,
        0.99
    ]:

        subset = (
            coverage_summary[
                (
                    coverage_summary["Model"]
                    == model
                )
                &
                (
                    coverage_summary["Confidence"]
                    == confidence
                )
            ]
            .copy()
        )

        validation_rows.append({

            "Model":
                model,

            "Confidence":
                confidence,

            "Portfolios Tested":
                len(subset),

            "Kupiec Passes":
                int(
                    subset[
                        "Kupiec Pass"
                    ].sum()
                ),

            "Independence Passes":
                int(
                    subset[
                        "Independence Pass"
                    ].sum()
                ),

            "Conditional Coverage Passes":
                int(
                    subset[
                        "Conditional Coverage Pass"
                    ].sum()
                )
        })


validation_scorecard = pd.DataFrame(
    validation_rows
)

validation_scorecard

,Model,Confidence,Portfolios Tested,Kupiec Passes,Independence Passes,Conditional Coverage Passes
0,Historical,0.95,5,3,4,5
1,Historical,0.99,5,5,5,5
2,Parametric Normal,0.95,5,4,4,4
3,Parametric Normal,0.99,5,0,4,0


In [63]:
latest_basel_summary = (
    latest_basel[
        [
            "Portfolio",
            "Model",
            "Exceptions",
            "Zone"
        ]
    ]
    .copy()
)

latest_basel_summary

,Portfolio,Model,Exceptions,Zone
953,Best GMVP,Historical,2,Green
2861,Best Maximum Sharpe,Historical,4,Green
4769,Best Minimum ES,Historical,2,Green
6677,Best Risk Parity,Historical,4,Green
8585,Equal Weight Benchmark,Historical,3,Green
1907,Best GMVP,Parametric Normal,4,Green
3815,Best Maximum Sharpe,Parametric Normal,5,Yellow
5723,Best Minimum ES,Parametric Normal,3,Green
7631,Best Risk Parity,Parametric Normal,11,Red
9539,Equal Weight Benchmark,Parametric Normal,6,Yellow


In [64]:
basel_latest_model_summary = (
    latest_basel_summary
    .groupby(
        [
            "Model",
            "Zone"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

basel_latest_model_summary

Zone,Green,Red,Yellow
Model,,,
Historical,5,0,0
Parametric Normal,2,1,2


In [65]:
validation_99 = (
    coverage_summary[
        coverage_summary[
            "Confidence"
        ]
        == 0.99
    ]
    [
        [
            "Portfolio",
            "Model",
            "Exceedances",
            "Observed Rate",
            "Kupiec p-value",
            "Kupiec Pass",
            "Christoffersen p-value",
            "Independence Pass",
            "Conditional Coverage p-value",
            "Conditional Coverage Pass"
        ]
    ]
    .copy()
)

validation_99

,Portfolio,Model,Exceedances,Observed Rate,Kupiec p-value,Kupiec Pass,Christoffersen p-value,Independence Pass,Conditional Coverage p-value,Conditional Coverage Pass
2,Best GMVP,Historical,15,0.012469,0.407251,True,0.538076,True,0.586862,True
3,Best GMVP,Parametric Normal,26,0.021613,0.000453,False,0.589453,True,0.001844,False
6,Best Maximum Sharpe,Historical,15,0.012469,0.407251,True,0.177659,True,0.285944,True
7,Best Maximum Sharpe,Parametric Normal,20,0.016625,0.034971,False,0.042063,False,0.013710,False
10,Best Minimum ES,Historical,15,0.012469,0.407251,True,0.538076,True,0.586862,True
11,Best Minimum ES,Parametric Normal,27,0.022444,0.000192,False,0.635224,True,0.000855,False
14,Best Risk Parity,Historical,15,0.012469,0.407251,True,0.177659,True,0.285944,True
15,Best Risk Parity,Parametric Normal,26,0.021613,0.000453,False,0.589453,True,0.001844,False
18,Equal Weight Benchmark,Historical,17,0.014131,0.175150,True,0.236405,True,0.197878,True
19,Equal Weight Benchmark,Parametric Normal,22,0.018288,0.009621,False,0.417161,True,0.025196,False


In [66]:
validation_99 = (
    validation_99
    .merge(
        latest_basel[
            [
                "Portfolio",
                "Model",
                "Exceptions",
                "Zone"
            ]
        ],
        on=[
            "Portfolio",
            "Model"
        ],
        how="left"
    )
)

validation_99 = (
    validation_99
    .rename(
        columns={
            "Exceptions":
                "Latest 250D Exceptions",

            "Zone":
                "Latest Basel Zone"
        }
    )
)

validation_99

,Portfolio,Model,Exceedances,Observed Rate,Kupiec p-value,Kupiec Pass,Christoffersen p-value,Independence Pass,Conditional Coverage p-value,Conditional Coverage Pass,Latest 250D Exceptions,Latest Basel Zone
0,Best GMVP,Historical,15,0.012469,0.407251,True,0.538076,True,0.586862,True,2,Green
1,Best GMVP,Parametric Normal,26,0.021613,0.000453,False,0.589453,True,0.001844,False,4,Green
2,Best Maximum Sharpe,Historical,15,0.012469,0.407251,True,0.177659,True,0.285944,True,4,Green
3,Best Maximum Sharpe,Parametric Normal,20,0.016625,0.034971,False,0.042063,False,0.013710,False,5,Yellow
4,Best Minimum ES,Historical,15,0.012469,0.407251,True,0.538076,True,0.586862,True,2,Green
5,Best Minimum ES,Parametric Normal,27,0.022444,0.000192,False,0.635224,True,0.000855,False,3,Green
6,Best Risk Parity,Historical,15,0.012469,0.407251,True,0.177659,True,0.285944,True,4,Green
7,Best Risk Parity,Parametric Normal,26,0.021613,0.000453,False,0.589453,True,0.001844,False,11,Red
8,Equal Weight Benchmark,Historical,17,0.014131,0.175150,True,0.236405,True,0.197878,True,3,Green
9,Equal Weight Benchmark,Parametric Normal,22,0.018288,0.009621,False,0.417161,True,0.025196,False,6,Yellow


In [67]:
validation_99[
    "Overall Validation Pass"
] = (
    validation_99[
        "Kupiec Pass"
    ]
    &
    validation_99[
        "Independence Pass"
    ]
    &
    validation_99[
        "Conditional Coverage Pass"
    ]
)

validation_99[
    [
        "Portfolio",
        "Model",
        "Overall Validation Pass",
        "Latest Basel Zone"
    ]
]

,Portfolio,Model,Overall Validation Pass,Latest Basel Zone
0,Best GMVP,Historical,True,Green
1,Best GMVP,Parametric Normal,False,Green
2,Best Maximum Sharpe,Historical,True,Green
3,Best Maximum Sharpe,Parametric Normal,False,Yellow
4,Best Minimum ES,Historical,True,Green
5,Best Minimum ES,Parametric Normal,False,Green
6,Best Risk Parity,Historical,True,Green
7,Best Risk Parity,Parametric Normal,False,Red
8,Equal Weight Benchmark,Historical,True,Green
9,Equal Weight Benchmark,Parametric Normal,False,Yellow


In [68]:
# Save Notebook 06 outputs

oos_portfolio_returns.to_csv(
    TABLE_DIR
    / "oos_portfolio_returns.csv"
)

oos_performance.to_csv(
    TABLE_DIR
    / "oos_performance_summary.csv"
)

oos_rankings.to_csv(
    TABLE_DIR
    / "oos_performance_rankings.csv"
)

var_backtests.to_csv(
    TABLE_DIR
    / "rolling_var_backtests.csv",
    index=False
)

exceedance_summary.to_csv(
    TABLE_DIR
    / "var_exceedance_summary.csv",
    index=False
)

kupiec_results.to_csv(
    TABLE_DIR
    / "kupiec_test_results.csv",
    index=False
)

christoffersen_results.to_csv(
    TABLE_DIR
    / "christoffersen_independence_results.csv",
    index=False
)

coverage_summary.to_csv(
    TABLE_DIR
    / "conditional_coverage_results.csv",
    index=False
)

basel_results.to_csv(
    TABLE_DIR
    / "basel_rolling_traffic_light.csv",
    index=False
)

validation_scorecard.to_csv(
    TABLE_DIR
    / "var_validation_scorecard.csv",
    index=False
)

validation_99.to_csv(
    TABLE_DIR
    / "var_validation_99_summary.csv",
    index=False
)

print(
    "Notebook 06 outputs saved successfully."
)

Notebook 06 outputs saved successfully.


In [69]:
assert returns_train.shape == (
    4309,
    7
)

assert returns_oos.shape == (
    1203,
    7
)

assert (
    returns_train.index.max()
    == pd.Timestamp("2021-12-31")
)

assert (
    returns_oos.index.min()
    == pd.Timestamp("2022-01-03")
)

assert len(shortlist) == 5

assert (
    shortlist[
        "Frozen Weights"
    ]
    .apply(
        lambda w:
        np.isclose(
            np.sum(w),
            1.0,
            atol=1e-6
        )
    )
    .all()
)

assert (
    len(oos_portfolio_returns)
    == len(returns_oos)
)

assert (
    oos_portfolio_returns
    .isna()
    .sum()
    .sum()
    == 0
)

assert len(kupiec_results) == 20
assert len(christoffersen_results) == 20
assert len(coverage_summary) == 20

assert (
    validation_scorecard[
        "Portfolios Tested"
    ]
    .eq(5)
    .all()
)

assert (
    basel_results[
        "Zone"
    ]
    .isin(
        [
            "Green",
            "Yellow",
            "Red"
        ]
    )
    .all()
)

print(
    "All Notebook 06 final "
    "validation checks passed."
)

All Notebook 06 final validation checks passed.


## Validation Interpretation

The 99% Historical VaR model performs substantially better than the
Parametric Normal model out of sample.

Historical VaR passes unconditional coverage, independence and conditional
coverage for all five candidate portfolios.

By contrast, the 99% Parametric Normal model produces too many exceedances and
fails unconditional and conditional coverage across the portfolio set.

This result is consistent with the earlier evidence of non-normal and
heavy-tailed return distributions and supports the use of non-parametric and
tail-focused risk measures for institutional downside-risk monitoring.